<h1>Research Paper Question Answering and Comparison System using RAG</h1>

<p>
This project implements a Retrieval-Augmented Generation (RAG) pipeline for answering questions from research papers and comparing information across multiple documents.
The system loads PDF files, converts them into searchable chunks, creates embeddings, stores them in a vector database, and retrieves relevant information to generate responses using a Large Language Model (LLM).
</p>

<h3>Objectives</h3>

<ul>
    <li>Load and process research papers in PDF format.</li>
    <li>Create embeddings for semantic search.</li>
    <li>Store document vectors using FAISS.</li>
    <li>Answer user questions from retrieved content.</li>
    <li>Compare information across multiple papers.</li>
    <li>Generate responses using a local LLM.</li>
</ul>

<h3>Technologies Used</h3>

<ul>
    <li>Python</li>
    <li>LangChain</li>
    <li>FAISS</li>
    <li>Sentence Transformers</li>
    <li>Ollama</li>
    <li>Llama 3.2</li>
</ul>

<h2>Step 1: Import Required Libraries</h2>

<p>
In this step, all the required libraries are imported for building the Research Paper Question Answering System.
These libraries help in loading PDF documents, splitting text into smaller chunks, generating embeddings,
storing vectors in a FAISS database, and interacting with a Large Language Model (LLM).
</p>

<h3>Purpose of the Imported Libraries</h3>

<ul>
    <li><b>os</b> – Used for file and directory management.</li>
    <li><b>PyPDFLoader</b> – Loads research papers from PDF files.</li>
    <li><b>RecursiveCharacterTextSplitter</b> – Splits large documents into smaller chunks for efficient retrieval.</li>
    <li><b>HuggingFaceEmbeddings</b> – Converts text into numerical vector representations.</li>
    <li><b>FAISS</b> – Stores and retrieves document embeddings using similarity search.</li>
    <li><b>ChatOllama</b> – Connects the application to a local Large Language Model.</li>
    <li><b>HumanMessage and SystemMessage</b> – Used to structure prompts and responses for the language model.</li>
</ul>

<p>
This step prepares all the tools required for document processing, semantic search, and answer generation.
</p>

In [15]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

from langchain_core.messages import (HumanMessage,SystemMessage)

<h2>Step 2: Load PDF Documents</h2>

<p>
In this step, a function is created to load a research paper in PDF format.
The function reads the PDF file and converts each page into a document object that can be processed later in the RAG pipeline.
</p>

<h3>What This Step Does</h3>

<ul>
    <li>Accepts the path of a PDF file as input.</li>
    <li>Loads the PDF using the PyPDFLoader utility.</li>
    <li>Extracts the content from all pages of the document.</li>
    <li>Stores the extracted pages as document objects.</li>
    <li>Displays information about the file being loaded.</li>
    <li>Returns the loaded documents for further processing.</li>
</ul>

<p>
This step serves as the entry point of the system, allowing research papers to be imported and prepared for text processing, chunking, and semantic retrieval.
</p>

In [16]:
def load_pdf(file_path):
    print(f"Loading: {os.path.basename(file_path)}")
    
    loader = PyPDFLoader(file_path)

    documents = loader.load()

    print(f"Loaded {len(documents)} page(s) from "f"{os.path.basename(file_path)}")
    return documents

## Step 3: Loading All PDF Files from a Folder

**Name: Load All PDFs and Combine Data**

In this step, we read all PDF files from a given folder and combine their content into one single dataset.

### What happens in this step:

- The program first looks inside the given folder and finds all files ending with **.pdf**
- If no PDF files are found, it stops and raises an error message
- Then it processes each PDF file one by one
- For every file, it uses the earlier function to load that PDF
- All pages from all PDFs are added into one combined list
- Finally, it prints the total number of pages loaded from all files

### Why this step is important:

This step helps us merge multiple documents into one dataset so we can work on them together for:
- RAG (Retrieval Augmented Generation)
- Embeddings creation
- Search and question-answering systems

### Output:

- A single combined list containing all PDF pages
- Total number of pages loaded from all PDFs

In [17]:
def load_all_pdfs(pdf_dir):
    all_documents = []

    pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith(".pdf")]

    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in '{pdf_dir}'")

    for filename in pdf_files:

        file_path = os.path.join(pdf_dir,filename)
        documents = load_pdf(file_path)
        all_documents.extend(documents)

    print(f"\nTotal pages loaded: "f"{len(all_documents)}")

    return all_documents

## Step 4: Splitting Text into Smaller Chunks

**Name: Text Chunking for Better Processing**

In this step, we break large documents into smaller, meaningful pieces so they can be processed efficiently.

### What happens in this step:

- The system takes all loaded documents as input
- It prepares a text splitter that divides text into smaller parts
- Each chunk has a fixed size (default around 1000 characters)
- Some overlap is kept between chunks (default 200 characters) to preserve context
- The text is split using natural separators like:
  - Paragraph breaks
  - New lines
  - Sentences
  - Spaces
- Finally, all documents are converted into multiple small chunks

### Why this step is important:

Large documents are hard for AI models to process directly. Splitting helps by:
- Making text easier to handle
- Preserving context between chunks
- Improving search and retrieval accuracy in RAG systems

### Output:

- A list of smaller text chunks
- Total number of chunks created is displayed

In [18]:
def split_text(documents,chunk_size=1000,chunk_overlap=200):
    print("Splitting documents...")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=[ "\n\n","\n","."," ",""]
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    
    return chunks

## Step 5: Creating the Embedding Model

**Name: Load Pretrained Embedding Model**

In this step, we load a pretrained model that converts text into numerical vectors (embeddings).

### What happens in this step:

- A Hugging Face embedding model is loaded
- The model used is: **all-MiniLM-L6-v2**
- This model converts sentences into vector form (numbers)
- These vectors help the system understand meaning, not just words

### Why this step is important:

In RAG systems, computers cannot understand raw text directly. So we convert text into embeddings to:
- Capture semantic meaning of text
- Compare similarity between queries and documents
- Improve search and retrieval accuracy

### Output:

- An embedding model object
- This model will be used later to convert chunks into vectors

In [19]:
def get_embedding_model():
    
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    return embedding_model

## Step 6: Creating and Saving the Vector Store

**Name: Build FAISS Vector Database**

In this step, we convert text chunks into embeddings and store them in a searchable vector database.

### What happens in this step:

- The system prints a message saying vector store creation has started
- It loads the embedding model from the previous step
- Each text chunk is converted into a numerical vector (embedding)
- These vectors are stored inside a FAISS database for fast similarity search
- The vector database is then saved locally to a folder (`db/faiss_db`)
- A confirmation message is printed after saving

### Why this step is important:

This step is the core of a RAG system because it allows:
- Fast searching of relevant text based on meaning
- Efficient storage of document embeddings
- Reuse of saved knowledge without recomputing embeddings every time

### Output:

- A FAISS vector store containing all document embeddings
- A saved local database for future use

In [20]:
def create_vector_store(chunks,persist_directory="db/faiss_db"):

    print("Creating vector store...")

    embedding_model = get_embedding_model()

    vectorstore = FAISS.from_documents(documents=chunks,embedding=embedding_model)

    vectorstore.save_local(persist_directory)

    print(f"Saved at {persist_directory}")

    return vectorstore

## Step 7: Loading an Existing Vector Store

**Name: Reload Saved FAISS Database**

In this step, we load an already created vector database so we don’t need to rebuild it again.

### What happens in this step:

- The embedding model is loaded again to ensure consistency
- The system looks for a saved FAISS vector database in the given folder
- The stored embeddings and index are loaded into memory
- The vector store becomes ready for searching again

### Why this step is important:

This step helps us avoid repeating expensive computations like:
- Re-creating embeddings
- Rebuilding the entire database

Instead, we can directly reuse saved data for:
- Faster startup
- Efficient querying
- Production-ready RAG systems

### Output:

- A ready-to-use FAISS vector store loaded from disk

In [21]:
def load_existing_vector_store(persist_directory="db/faiss_db"):

    embedding_model = get_embedding_model()

    vectorstore = FAISS.load_local(
        folder_path=persist_directory,
        embeddings=embedding_model,
        allow_dangerous_deserialization=True
    )
    
    return vectorstore

## Step 8: Retrieving Relevant Chunks from Vector Store

**Name: Semantic Search / Document Retrieval**

In this step, we search the vector database to find the most relevant text chunks for a given user query.

### What happens in this step:

- The vector store is converted into a retriever (search system)
- The retriever is configured to return the top **k** most relevant results (default is 4)
- The user query is passed into the retriever
- The system finds chunks that are most similar in meaning to the query
- These relevant chunks are returned as output

### Why this step is important:

This is the core “retrieval” part of RAG systems. It helps to:
- Find the most meaningful context from large documents
- Avoid scanning the entire dataset manually
- Provide focused information for the language model

### Output:

- A list of top-k relevant document chunks related to the query

In [22]:
def retrieve_chunks(vectorstore,query,k=4):

    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    relevant_docs = retriever.invoke(query)

    return relevant_docs

## Step 9: Retrieving Chunks Per Paper (Source-Wise Retrieval)

**Name: Grouped Semantic Search by Document Source**

In this step, we retrieve relevant text chunks but also group them based on their original PDF file.

### What happens in this step:

- The vector store is converted into a retriever
- It first fetches a larger set of relevant chunks (e.g., top 50 results)
- Each retrieved chunk contains metadata about its source file (PDF name)
- The system groups chunks based on their source document
- From each document, it selects only a limited number of chunks (default is 3 per paper)
- This ensures balanced representation from multiple PDFs

### Why this step is important:

This step improves RAG performance when multiple documents are involved by:
- Preventing one document from dominating the results
- Ensuring diversity across different sources
- Keeping context balanced across multiple papers or PDFs
- Improving fairness in retrieval across datasets

### Output:

- A dictionary where:
  - Key = PDF filename
  - Value = list of relevant chunks from that PDF

In [23]:
def retrieve_chunks_per_paper(vectorstore,query,k_per_paper=3):

    retriever = vectorstore.as_retriever(search_kwargs={"k": 50})
    all_docs = retriever.invoke(query)
    paper_docs = {}

    for doc in all_docs:
        source = os.path.basename(doc.metadata.get("source","unknown"))

        if source not in paper_docs:
            paper_docs[source] = []

        if len(paper_docs[source]) < k_per_paper:
            paper_docs[source].append(doc)

    return paper_docs

## Step 10: Generating the Final Answer

**Name: LLM Answer Generation using Retrieved Context**

In this step, we use a language model to generate the final answer based only on the retrieved documents.

### What happens in this step:

- The system takes all relevant documents retrieved in the previous step
- It combines them into a single context string
- Each chunk is labeled with its source file name for reference
- A final input is created containing:
  - The user’s question
  - The retrieved document context
- A language model (LLM) is loaded (**Llama 3.2 1B via Ollama**)
- The model is instructed to answer only using the provided documents
- The response is generated and returned as the final answer

### Why this step is important:

This is the **generation part of RAG**, where:
- Retrieval provides knowledge
- The LLM converts that knowledge into a natural answer
- It reduces hallucination by restricting the model to given context

### Output:

- A final natural language answer
- The answer is strictly based on the provided documents

In [24]:
def generate_answer(relevant_docs,query):

    context = "\n\n".join([
        f"[Source: {os.path.basename(doc.metadata.get('source','unknown'))}]\n"
        f"{doc.page_content}"
        for doc in relevant_docs
    ])

    combined_input = f"""
Question:
{query}

Documents:
{context}
"""
    model = ChatOllama(model="llama3.2:1b")

    messages = [SystemMessage(content=(
                "Answer only from "
                "the provided documents.")),
        HumanMessage(content=combined_input)]

    result = model.invoke(messages)

    return result.content

## Step 11: Comparing Multiple Documents

**Name: Multi-Paper Comparison using LLM**

In this step, we compare information across multiple documents or research papers using a language model.

### What happens in this step:

- The system receives grouped documents (organized by paper/source)
- For each paper:
  - All its relevant chunks are combined into a single text block
  - The paper name is added as a header for clarity
- All paper sections are merged into one final context
- A combined input is created that includes:
  - The user’s question
  - All papers' content
- A language model (Llama 3.2 1B via Ollama) is used
- The model is instructed to compare the research papers
- The final comparative answer is generated

### Why this step is important:

This step helps when working with multiple documents by:
- Allowing side-by-side comparison of research papers
- Extracting similarities and differences
- Helping in literature review tasks
- Supporting academic and research analysis

### Output:

- A detailed comparison-based answer
- Insights across multiple documents instead of just one

In [25]:
def compare_documents(paper_docs,query):
    
    paper_sections = []
    
    for paper_name, docs in paper_docs.items():
        
        paper_text = "\n".join([doc.page_content
            for doc in docs])
        section = (f"=== {paper_name} ===\n"
            f"{paper_text}")

        paper_sections.append(section)
    all_context = "\n\n".join(paper_sections)

    combined_input = f"""
Question:
{query}

Papers:
{all_context}
"""

    model = ChatOllama(model="llama3.2:1b")

    messages = [SystemMessage(content=("Compare research papers.") ),
        HumanMessage(content=combined_input)]

    result = model.invoke(messages)

    return result.content

## Step 12: Detecting Comparison Queries

**Name: Query Type Detection (Comparison vs Normal Query)**

In this step, we check whether the user is asking for a comparison between multiple documents or just a normal question.

### What happens in this step:

- A list of comparison-related keywords is defined
- The user query is converted into lowercase for easy matching
- The system checks if any keyword appears in the query
- If a match is found, it is treated as a comparison question
- Otherwise, it is treated as a normal retrieval question

### Why this step is important:

This step helps the system decide:
- Whether to use simple RAG (single document answer)
- Or multi-document comparison logic

It improves:
- Accuracy of responses
- Proper routing of queries
- Better handling of research-based questions

### Output:

- Returns **True** if query is a comparison type
- Returns **False** otherwise

In [26]:
def is_comparison_query(query):

    keywords = [
        "compare",
        "contrast",
        "agree",
        "disagree",
        "difference",
        "similar",
        "contradict",
        "both papers",
        "all papers"
    ]

    query = query.lower()

    return any(
        keyword in query
        for keyword in keywords)

In [27]:
pdf_dir = "papers"

persist_directory = "db/faiss_db"

if os.path.exists(persist_directory):
    vectorstore = (load_existing_vector_store(
            persist_directory))
else:

    documents = load_all_pdfs(pdf_dir)

    chunks = split_text(documents)

    vectorstore = create_vector_store(chunks,persist_directory)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
query = input("Ask a question: ")

if is_comparison_query(query):
    
    paper_docs = (retrieve_chunks_per_paper(vectorstore,query))

    answer = compare_documents(paper_docs,query)

else:

    relevant_docs = (retrieve_chunks(vectorstore,query))

    answer = generate_answer(relevant_docs,query)

print(answer)

Ask a question:  Compare BERT and GPT-3 methodologies and identify their key differences.


**BERT Methodology:**

* **Training setup:** BERT is trained on a large text corpus, including BooksCorpus and Wikipedia.
* **Architecture:** BERT uses a bidirectional Transformer model, where both left-to-right and right-to-left models are used in all layers.
* **Fine-tuning approach:** Unlike other pre-trained models like GPT-3, BERT fine-tunes its representations on the training data to adapt to downstream tasks.

**GPT-3 Methodology:**

* **Training setup:** GPT-3 is trained with a left-to-right Transformer model, using sentence separator and classifier tokens.
* **Architecture:** While GPT-3 uses a left-to-right Transformer architecture, it does not fine-tune its representations on the training data like BERT.
* **Fine-tuning approach:** Unlike BERT, GPT-3 is trained with pre-trained word embeddings and sentence embedding layers, which are fine-tuned during training.

**Key differences:**

1. **Architecture:** BERT uses a bidirectional Transformer model in all layers, while GPT-3 